In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

NOTEBOOK_DIR = Path.cwd() / "notebooks" if (Path.cwd() / "notebooks").exists() else Path.cwd()
sys.path.insert(0, str(NOTEBOOK_DIR))
from pipeline_helpers import ARTIFACT_DIR, DATA_PATH, SEED

print(f"Data: {DATA_PATH}")
print(f"Artifacts: {ARTIFACT_DIR}")
print(f"Reproducibility seed: {SEED}")
from pipeline_helpers import phase2
result = phase2()
display(result["summary"])

Data: D:\4th Semester\DS4105 - Capstone Project in Data Science II\reburn\data\raw\europe_cereal_yield_climate_1990_2022.csv
Artifacts: D:\4th Semester\DS4105 - Capstone Project in Data Science II\reburn\artifacts\cereal_yield_pipeline
Reproducibility seed: 42


,metric,value
0,correlation,1.000000
1,exact_match_abs_error_lt_1e-9,67.000000
2,within 0.1 yield units,3199.000000
3,median absolute error,0.024915
4,95th percentile absolute error,0.048430
5,maximum absolute error,23.000000


## Production/area identity

For these source fields, the natural yield conversion is `PRODUCTION_QUANTITY / AREA_HARVESTED × 1000`. The result is rounded rather than identical for every row: inspect the error summary and largest deviations below. Because the ratio still nearly reproduces YIELD, both accounting fields are excluded from predictors.

In [2]:
display(result["largest_differences"])
display(result["leakage_columns_dropped"])

,AREA,ITEM,YEAR,YIELD,AREA_HARVESTED,PRODUCTION_QUANTITY,derived_yield,absolute_error
1705,Luxembourg,Maize (corn),2016,6697.0,125.0,840.00,6720.000000,23.000000
1706,Luxembourg,Maize (corn),2017,8597.2,81.0,695.00,8580.246914,16.953086
1756,Malta,Barley,2011,4170.1,510.0,2128.82,4174.156863,4.056863
1760,Malta,Barley,2015,4179.9,522.0,2180.12,4176.475096,3.424904
1743,Malta,Barley,1998,3241.0,525.0,1700.00,3238.095238,2.904762
1803,Montenegro,Barley,2018,2736.7,395.0,1082.10,2739.493671,2.793671
1820,Montenegro,Maize (corn),2018,4250.5,645.0,2743.30,4253.178295,2.678295
1819,Montenegro,Maize (corn),2017,4220.1,642.0,2707.60,4217.445483,2.654517
1807,Montenegro,Barley,2022,2620.2,311.0,814.10,2617.684887,2.515113
1824,Montenegro,Maize (corn),2022,4539.8,620.0,2813.30,4537.580645,2.219355


,column,reason
0,AREA_HARVESTED,Used in arithmetic identity for YIELD
1,PRODUCTION_QUANTITY,Used in arithmetic identity for YIELD


3,199 of 3,245 rows are within 0.1 yield units of the production/area identity; the median absolute difference is 0.025. The maximum difference is 23, so the relationship is not exact for every record, but it remains strong enough to constitute target leakage.

The two accounting fields are excluded from all predictors even if rounding prevents an exact row-by-row match. The leakage-free data are saved to `phase2_no_leakage.pkl`.